## Step 6: Running the Application

In [1]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()

slice = fablib.get_slice(name="P4DPDKJournal")

servers = []

servers.append(slice.get_node(name="server1"))     
servers.append(slice.get_node(name="server2"))
servers.append(slice.get_node(name="server3"))

server1 = servers[0]
server2 = servers[1]
server3 = servers[2]

User: choueiri@email.sc.edu bastion key is valid!
Configuration is valid


In [2]:
server1.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fee2:7e98'

In [3]:
server2.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fe85:74c0'

In [4]:
server3.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3080:f816:3eff:fee7:1f1e'

### P4-DPDK Reflector

In [12]:
stdout, stderr = server2.execute(f'mkdir reflector')

In [13]:
server2.upload_file('scripts/code/reflector.p4','reflector/reflector.p4')
stdout, stderr = server2.execute(f'cd reflector && sudo p4c-dpdk --arch=pna reflector.p4 -o reflector.spec')
stdout, stderr = server2.execute(f'ls reflector/')

reflector.p4
reflector.spec


In [14]:
server2.upload_file('scripts/code/reflector.cli','reflector/reflector.cli')
server2.upload_file('scripts/code/ethdev0.io','reflector/ethdev0.io')
server2.upload_file('scripts/code/ethdev1.io','reflector/ethdev1.io')
server2.upload_file('scripts/code/ethdev2.io','reflector/ethdev2.io')
server2.upload_file('scripts/code/ethdev3.io','reflector/ethdev3.io')

<SFTPAttributes: [ size=237 uid=1000 gid=1000 mode=0o100664 atime=1787243082 mtime=1787243082 ]>

In [15]:
threads = []

for server in servers:
    threads.append(server.execute_thread(f' sudo sh -c  "echo 1024 > /sys/kernel/mm/hugepages/hugepages-2048kB/nr_hugepages"'))

for thread in threads:
    thread.result()

## E-switch

### Check interfaces

In [3]:
stdout, stderr = server2.execute(f'lspci | grep -i mellanox')

07:00.0 Ethernet controller: Mellanox Technologies MT28908 Family [ConnectX-6]
08:00.0 Ethernet controller: Mellanox Technologies MT28908 Family [ConnectX-6]


In [4]:
stdout, stderr = server2.execute(f'lspci -D | grep -i mellanox')

0000:07:00.0 Ethernet controller: Mellanox Technologies MT28908 Family [ConnectX-6]
0000:08:00.0 Ethernet controller: Mellanox Technologies MT28908 Family [ConnectX-6]


In [6]:
stdout, stderr = server2.execute(f'ethtool -i enp7s0np0 | grep bus-info')
stdout, stderr = server2.execute(f'ethtool -i enp8s0np1 | grep bus-info')

bus-info: 0000:07:00.0
bus-info: 0000:08:00.0


In [7]:
stdout, stderr = server2.execute(f'dpdk-devbind.py --status')


Network devices using kernel driver
0000:03:00.0 'Virtio network device 1041' if=enp3s0 drv=virtio-pci unused=vfio-pci *Active*
0000:07:00.0 'MT28908 Family [ConnectX-6] 101b' if=enp7s0np0 drv=mlx5_core unused=vfio-pci *Active*
0000:08:00.0 'MT28908 Family [ConnectX-6] 101b' if=enp8s0np1 drv=mlx5_core unused=vfio-pci *Active*

No 'Baseband' devices detected

No 'Crypto' devices detected

No 'DMA' devices detected

No 'Eventdev' devices detected

No 'Mempool' devices detected

No 'Compress' devices detected

Misc (rawdev) devices using kernel driver
0000:04:00.0 'Virtio block device 1042' drv=virtio-pci unused=vfio-pci 

No 'Regex' devices detected

No 'ML' devices detected


In [7]:
stdout, stderr = server2.execute(f'sudo devlink dev eswitch show pci/0000:08:00.0')

pci/0000:08:00.0: mode legacy inline-mode none encap enable


mode legacy — This is the key one. The eswitch is in legacy mode, not switchdev mode. In legacy mode, there are no VF/SF representor netdevs exposed to the host; SR-IOV VFs (if any) are managed through the old-style embedded switch model where the PF driver handles VF policy internally (MAC/VLAN filtering, rate limiting, etc.) rather than exposing a Linux representor you can attach tc/rte_flow rules to per-VF.

inline-mode none — Inline mode controls whether the eswitch requires L2 header fields to be duplicated inline in send descriptors for steering decisions on VF traffic. This matters for older ConnectX‑3/4 style embedded switch modes with certain SR-IOV configs; none means it's not needed/active here, which is normal for ConnectX‑6 in legacy mode with no VFs currently provisioned.

encap enable — Hardware encap/decap (VXLAN etc.) offload capability is enabled at the eswitch level. Not directly relevant to your ACL-drop use case, just means tunnel offload capability is on.

### Make sure pipeline is not running

Confirm whether your P4-DPDK pipeline is still running

In [10]:
stdout, stderr = server2.execute(f'ps aux | grep -i pipeline')

ubuntu     43055  0.0  0.0   8616  3060 ?        Ss   21:11   0:00 bash -c ps aux | grep -i pipeline
ubuntu     43057  0.0  0.0   8160  2564 ?        R    21:11   0:00 grep -i pipeline


Verify no process still has the device's ibverbs context open

In [12]:
stdout, stderr = server2.execute(f'sudo fuser -v /dev/infiniband/*')

Double check which PF is the one your pipeline actually uses

In [14]:
stdout, stderr = server2.execute(f'sudo ethtool -i enp7s0np0 | grep bus-info')
stdout, stderr = server2.execute(f'sudo ethtool -i enp8s0np1 | grep bus-info')

bus-info: 0000:07:00.0
bus-info: 0000:08:00.0


### Take a snapshot of the configuration

In [ ]:
stdout, stderr = server2.execute(f'ip link show > /tmp/before_links.txt')

In [ ]:
stdout, stderr = server2.execute(f'dpdk-devbind.py --status > /tmp/before_devbind.txt')

In [ ]:
stdout, stderr = server2.execute(f'sudo devlink dev eswitch show pci/0000:08:00.0')

### Enable switchdev mode on the PF

In [15]:
stdout, stderr = server2.execute(f'sudo devlink dev eswitch set pci/0000:08:00.0 mode switchdev')

In [16]:
stdout, stderr = server2.execute(f'sudo devlink dev eswitch show pci/0000:08:00.0')

pci/0000:08:00.0: mode switchdev inline-mode none encap enable


In [17]:
stdout, stderr = server2.execute(f'ip link show')

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: enp3s0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc fq_codel state UP mode DEFAULT group default qlen 1000
    link/ether fa:16:3e:85:74:c0 brd ff:ff:ff:ff:ff:ff
3: enp7s0np0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 8982 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:00:00:00:00:21 brd ff:ff:ff:ff:ff:ff
4: enp8s0np1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 8982 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:00:00:00:00:22 brd ff:ff:ff:ff:ff:ff


In [18]:
stdout, stderr = server2.execute(f'cat /tmp/before_links.txt')

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: enp3s0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc fq_codel state UP mode DEFAULT group default qlen 1000
    link/ether fa:16:3e:85:74:c0 brd ff:ff:ff:ff:ff:ff
3: enp7s0np0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 8982 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:00:00:00:00:21 brd ff:ff:ff:ff:ff:ff
4: enp8s0np1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 8982 qdisc mq state UP mode DEFAULT group default qlen 1000
    link/ether 00:00:00:00:00:22 brd ff:ff:ff:ff:ff:ff


### Launch testpmd

In [16]:
stdout, stderr = server2.execute(f'sudo devlink dev eswitch show pci/0000:08:00.0')

pci/0000:08:00.0: mode switchdev inline-mode none encap enable


### Blacklist the flow

### Test if flow is dropped

### Push rule from pipeline

In [22]:
# server2.download_file('pipeline_backup.zip','pipeline_backup.zip')

In [2]:
server2.upload_file('scripts/cli.c','/home/ubuntu/dpdk/examples/pipeline/cli.c')
server2.upload_file('scripts/obj.c','/home/ubuntu/dpdk/examples/pipeline/obj.c')
server2.upload_file('scripts/obj.h','/home/ubuntu/dpdk/examples/pipeline/obj.h')

<SFTPAttributes: [ size=1165 uid=1000 gid=1000 mode=0o100664 atime=1789701769 mtime=1790089961 ]>

In [4]:
stdout, stderr = server2.execute(f'cd /home/ubuntu/dpdk/examples/pipeline && sudo make')

ln -sf pipeline-shared build/pipeline
